In [3]:
from rag_helper import RAGbase
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
openai_client=OpenAI()

In [4]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [5]:
INSTRUCTIONS = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

#question="I Just discovered the course, can I still join it?"


#messages= [
#    {'role':"developer", 'content':INSTRUCTIONS},
#    {"role": "user", "content": question}
#]



In [6]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [7]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [14]:
response=openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)


In [15]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"can I still join course discovered late enrollment join course FAQ"}', call_id='call_0JHPaVddE22lkn9l9lymwGZ5', name='search', type='function_call', id='fc_09402c504d04fc43006a3beb0d957881a1b0d0838496980318', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"late join course enrollment access after course started FAQ"}', call_id='call_O3gBOz73zqtKfTrtrh4X1zUP', name='search', type='function_call', id='fc_09402c504d04fc43006a3beb0d959081a1a0523756b8cace2a', namespace=None, status='completed')]

In [16]:
response.output_text

''

In [8]:
import json

def make_call(call):
    args = json.loads(call.arguments)

    if call.name=="search":
        result=search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type":"function_call_output",
        "call_id":call.call_id,
        "output": result_json
    }

In [19]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

function_call: search {"query":"can I still join course discovered late enrollment join course FAQ"}
function_call: search {"query":"late join course enrollment access after course started FAQ"}


In [20]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user',
  'content': 'I Just discovered the course, can I still join it?'},
 ResponseFunctionToolCall(arguments='{"query":"can I still join course discovered late enrollment join course FAQ"}', call_id='call_0JHPaVddE22lkn9l9lymwGZ5', name='search', type='function_call', id='fc_09402c504d04fc43006a3beb0d957881a1b0d0838496980318', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"late join course enrollment access after course started FAQ"}', call_

In [31]:
question="Can I use other LLMs?"

In [32]:
messages = [
    {'role': 'developer', 'content': INSTRUCTIONS},
    {'role': 'user', 'content': question}
]

In [33]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [34]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"other LLMs use policy course FAQ"}', call_id='call_MUx1MeIqplspLCe3aTIu8DAV', name='search', type='function_call', id='fc_0902b7fb5c4f7488006a3bfaa007b481a2bfd4aea0e552aa13', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"can I use other LLMs assignment policy FAQ"}', call_id='call_ACXVexnLISAafh3BWPqFfatW', name='search', type='function_call', id='fc_0902b7fb5c4f7488006a3bfaa007c881a298eef6abb5c68bf5', namespace=None, status='completed')]

In [35]:
messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"other LLMs use policy course FAQ"}
function_call: search {"query":"can I use other LLMs assignment policy FAQ"}


In [36]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — in this course, you can use other LLMs/providers. The FAQ says the recommended model is **not mandatory**. You may use:

- **OpenAI**
- **Gemini**
- **Groq**
- **OpenRouter**
- **Azure OpenAI**
- **local models**
- or another provider

A couple of notes:
- You may need to **adapt the code** because response formats, tool schemas, and tokenizers can differ.
- The homework is designed so you **shouldn’t need a paid service**.
- For provider ideas, the course points to the **OpenAI API alternatives** list in the repo.

If you want, I can also help you figure out **which LLM/provider is easiest to use for a specific homework or project**. Any other areas you want to explore?


In [37]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'Can I use other LLMs?'},
 ResponseFunctionToolCall(arguments='{"query":"other LLMs use policy course FAQ"}', call_id='call_MUx1MeIqplspLCe3aTIu8DAV', name='search', type='function_call', id='fc_0902b7fb5c4f7488006a3bfaa007b481a2bfd4aea0e552aa13', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"can I use other LLMs assignment policy FAQ"}', call_id='call_ACXVexnLISAafh3BWPqFfatW', name='search', type='function_call', id='fc_09

In [9]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [10]:
agent_loop(INSTRUCTIONS,"How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run install local Ollama"}
function_call: search {"query":"Ollama local run FAQ install setup"}
iteration #2...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - **macOS**: download the `.pkg` from [ollama.com/download](https://ollama.com/download)
   - **Windows**: download the `.msi`
   - **Linux**:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This downloads the model and opens a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should see a response from Ollama.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```

   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )

   print(response['message']['content'])
   ```

If yo

'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - **macOS**: download the `.pkg` from [ollama.com/download](https://ollama.com/download)\n   - **Windows**: download the `.msi`\n   - **Linux**:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model and opens a local chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should see a response from Ollama.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a connection error, restart the server with:\n```bash\nollama serve\n```\nor in a notebook:\n```bash\n!nohup ollama serve > nohup.out

In [11]:
instructions2 = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [12]:
agent_loop(instructions2, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening"}
iteration #3...
function_call: search {"query":"queen gambit meaning course FAQ"}
iteration #4...
ASSISTANT:
I couldn’t find any course FAQ entry for “queen gambit,” so I can’t confirm what it refers to from the course materials.

If you meant a course topic, please share a bit more context or the exact term, and I can look again. Are there other areas you want to explore?


'I couldn’t find any course FAQ entry for “queen gambit,” so I can’t confirm what it refers to from the course materials.\n\nIf you meant a course topic, please share a bit more context or the exact term, and I can look again. Are there other areas you want to explore?'

ToyAIKit Approach

In [13]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [14]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [15]:
def search(query:str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question":3.0,"section":0.5},
        filter_dict={"course":"llm-zoomcamp"}
    )

In [16]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [17]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course FAQ.'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [ ]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [22]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions2,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)


In [23]:
question="Can I use other LLMs?"

In [24]:
result=runner.loop(
    prompt=question,
    callback=callback
)

-> Response received


-> Response received


-> Response received


In [25]:
result.cost

CostInfo(input_cost=Decimal('0.003018'), output_cost=Decimal('0.0012825'), total_cost=Decimal('0.0043005'))

In [27]:
#continuing conversation
result2=runner.loop(
    prompt="Would Gemini be cheaper?",
    previous_messages=result.all_messages,
    callback=callback
)

-> Response received


-> Response received


In [28]:
result2.cost

CostInfo(input_cost=Decimal('0.005034'), output_cost=Decimal('0.0006705'), total_cost=Decimal('0.0057045'))